# Reproduce the 0.83460 submission — Detect Suspicious Value Transfers in Poker

This notebook deterministically regenerates the best submission (`evidence_familycond_submission.csv`, public LB **0.83460**).

**What it does.** The submission is the family-conditional evidence stack: it takes the 0.83185 base (`seq_evidence_submission.csv`) and swaps in evidence hands ranked by a per-family LightGBM ranker (directed_transfer / soft_play / coordinated_isolation), keeping the `risk_score` and `predicted_behavior` columns **byte-identical** to the base. Only the 5 `evidence_hand_*` columns change.

**Verified.** Re-running the generator against the on-disk caches reproduces the banked CSV **byte-for-byte** (SHA256 `2ba41953a89fa752...`).

**Inputs required (all under `outputs/poker_collusion/`):**
- `seq_evidence_submission.csv` — the 0.83185 base (source of risk + behavior columns)
- `hosen42_step3/dev_pair_features.parquet`, `eval_pair_features.parquet` — pair-level features (family heads)
- `hosen42_step3/eval_hand_features.parquet` — per-hand features for eval (evidence ranking)
- `hosen42_step3/edge/seq/seq_ctx_emb_dev.parquet`, `seq_ctx_emb_eval.parquet` — cached sequence-encoder context embeddings

**Runtime:** ~5 min (LightGBM CPU + cached GPU embeddings). No GPU strictly required for this step since embeddings are cached.

## 1. Environment check — confirm all input artifacts are present

In [ ]:
from pathlib import Path

OUT = Path('outputs/poker_collusion')
STEP3 = OUT / 'hosen42_step3'
SEQ = STEP3 / 'edge' / 'seq'

required = [
    OUT / 'seq_evidence_submission.csv',
    STEP3 / 'dev_pair_features.parquet',
    STEP3 / 'eval_pair_features.parquet',
    STEP3 / 'eval_hand_features.parquet',
    SEQ / 'seq_ctx_emb_dev.parquet',
    SEQ / 'seq_ctx_emb_eval.parquet',
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f'MISSING inputs: {missing}'
print('All input artifacts present.')
for p in required:
    print(f'  {p.stat().st_size/1e6:8.1f} MB  {p}')

## 2. Generate the submission

Runs `anchor_repro.evidence_familycond_submit`, which:
1. Fits the 3 hosen42 family heads on labeled dev pairs and takes the prior-normalized argmax per eval pair.
2. Trains 3 per-family evidence rankers on all dev positives (`HAND_FEATS` + context embeddings, target `is_evidence`).
3. Streams the 9.65M eval hands, scores each by its pair's family ranker, takes top-5 per pair (host ordering).
4. Swaps the evidence columns into the 0.83185 base and asserts `risk_score`/`predicted_behavior` are unchanged.

Output: `outputs/poker_collusion/evidence_familycond_submission.csv`.

In [ ]:
import subprocess, sys
res = subprocess.run([sys.executable, '-u', '-m', 'anchor_repro.evidence_familycond_submit'],
                     capture_output=True, text=True)
print(res.stdout[-3000:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-2000:])
assert res.returncode == 0, 'generation failed'

## 3. Validate schema and value ranges

In [ ]:
import pandas as pd
sub = pd.read_csv(OUT / 'evidence_familycond_submission.csv')

expected_cols = ['pair_id', 'risk_score', 'predicted_behavior',
                 'evidence_hand_1', 'evidence_hand_2', 'evidence_hand_3',
                 'evidence_hand_4', 'evidence_hand_5']
assert list(sub.columns) == expected_cols, sub.columns.tolist()
assert len(sub) == 112540, len(sub)
assert sub['risk_score'].between(0, 1).all()
assert sub['pair_id'].is_unique
assert set(sub['predicted_behavior'].unique()) <= {'none','directed_transfer','soft_play','coordinated_isolation'}
print('Schema OK:', len(sub), 'rows')
print('risk_score range:', float(sub.risk_score.min()), '->', float(sub.risk_score.max()))
print('behavior counts:', sub.predicted_behavior.value_counts().to_dict())
sub.head(3)

## 4. Byte-match check against the banked 0.83460 (optional integrity gate)

The regenerated file should match the banked SHA256 prefix `2ba41953a89fa752`. If it differs, the caches or
code changed since the banked run — investigate before submitting.

In [ ]:
import hashlib
digest = hashlib.sha256(open(OUT / 'evidence_familycond_submission.csv', 'rb').read()).hexdigest()
print('regenerated sha256:', digest[:16])
print('expected (banked): ', '2ba41953a89fa752')
print('BYTE-MATCH' if digest.startswith('2ba41953a89fa752') else 'DIFFERS — investigate before submitting')

## 5. Submit

```bash
kaggle competitions submit -c detect-suspicious-value-transfers-in-poker \
  -f outputs/poker_collusion/evidence_familycond_submission.csv \
  -m "family-conditional evidence stack (0.83460 repro)"
```
(The CLI may print an exit-1 warning but the submission still registers.)